In [1]:
import pandas as pd
import numpy as np
import os
import glob as glob
import sys
import json

parent_dir = os.path.abspath(os.path.join(os.path.dirname(os.getcwd())))
sys.path.append(parent_dir)
import cpg_harmonizer
import s3_loader
import harmonize_checker

In [2]:
%load_ext autoreload
%autoreload 2

# Enter Identifiers, Per-Dataset Metadata

In [3]:
in_staging = True # True or False
profile = "CPGnew" # enter AWS profile if in_staging is True, otherwise None

In [4]:
project_name = "cpg0048-bsc" # e.g. cpg0016-jump
source_list = ['axiom'] # e.g. ['broad'] or ['source_1','source_2']
output_parent_directory = "/Users/eweisbar/Desktop/cpg0048_metadata_output"

# typically, per-dataset metadata
# if not dataset-wide, delete from here and create conditional entry below
# if unknown, comment out
per_dataset_manual = {
    'Plate_Size':384,
    'CP_Version':'v3', # e.g. "v1"
    'DOI_to_Cite':'',
    'Year_Imaged': 2024,
    'Cell_Line_Name':'BioIVT LIVERPOOL',
    'Cell_Line_Type':'primary human hepatocyte',
    'Cell_Line_Organism':'Homo sapiens',
    'Cell_Line_Modification':'None',
    'Microscope_Name':'Revvity Operetta',
    'Microscope_Binning': 1,
    'Microscope_Modality':'Widefield',
    'Microscope_Objective_Magnification':40,
    'Microscope_Objective_NA':1.1,
    'Microscope_Pixel_Size': .1483,
    'Image_Bit_Depth': 16,
    'Image_Size_X':2160,
    'Image_Size_Y':2160,
    'Timepoint_Primary_Treatment':44,
    'Treatment_Category':'Compound',
}

In [5]:
# excitation/emission values used for acquisition
# if value is unknown, define value with np.nan
# if excitation is with a laser (negligible width), 'ex_width' = "None"

# 'fluorophore' is fluorophore conjugated to label or dye variant
# "None" if none, np.nan if unknown
ex_em_fluor_dict = {
    'DNA':{'ex_peak':365,
           'ex_width':30,
           'em_peak':465,
           'em_width':70,
           'fluorophore':'33342'},
    'ER':{'ex_peak':475,
          'ex_width':30,
           'em_peak':525,
           'em_width':50,
           'fluorophore':'PhenoVue Fluor 488'},
    'RNA':{'ex_peak':475,
           'ex_width':30,
           'em_peak':610,
           'em_width':80,
           'fluorophore':'PhenoVue Fluor 512'},
    'AGP':{'ex_peak':550,
           'ex_width':30,
           'em_peak':610,
           'em_width':80,
           'fluorophore':'PhenoVue Fluor 555, PhenoVue Fluor 568'},
    'Mito':{'ex_peak':630,
            'ex_width':30,
           'em_peak':708,
           'em_width':105,
           'fluorophore':'PhenoVue Fluor 641'},
    'Brightfield':{'ex_peak':740,
            'ex_width':np.nan,
           'em_peak':722,
           'em_width':75,
           'fluorophore':'None'}
}

# Join CPG Metadata

In [6]:
if len(source_list) == 1:
    output_directory = os.path.join(output_parent_directory, project_name, source_list[0], "workspace", "metadata_harmonized")
else:
    output_directory = os.path.join(output_parent_directory, project_name, "all", "workspace", "metadata_harmonized")
if not os.path.exists(output_directory):
    os.makedirs(output_directory, exist_ok=True)

metadata_paths = []
load_paths = []
for src in source_list:
    metadata_paths.extend(s3_loader.parse_s3_folder(f"{project_name}/{src}/workspace/metadata/platemaps/", in_staging=in_staging, profile=profile))
    load_paths.extend(s3_loader.parse_s3_folder(f"{project_name}/{src}/workspace/load_data_csv/", in_staging=in_staging, profile=profile))

project = cpg_harmonizer.Project(output_directory, "../output_structure.json", project_name=project_name)

In [7]:
load_data_csvs = []
list_batch_names_from_load_data = []
for path in load_paths:
    if path.endswith("load_data.csv"):
        load_data_csvs.append(s3_loader.read_s3_file(path, sep = ",", in_staging=in_staging, profile=profile))
        list_batch_names_from_load_data.append(path.split("/")[-3])

In [8]:
platemaps = []
barcode_platemap_csvs = []
list_batch_names_from_barcode_platemaps = []
list_platemap_names = []
list_batch_names_from_platemaps = []
external_tsv = []

for path in metadata_paths:
    if path.endswith("barcode_platemap.csv"):
        barcode_platemap_csvs.append(s3_loader.read_s3_file(path, sep = ",",in_staging=in_staging, profile=profile))
        list_batch_names_from_barcode_platemaps.append(path.split("/")[-2])
    if '/platemap/' in path and path.endswith(".txt"):
        platemaps.append(s3_loader.read_s3_file(path, sep = "\t", in_staging=in_staging, profile=profile))
        list_platemap_names.append(path.split("/")[-1].split(".")[0])
        list_batch_names_from_platemaps.append(path.split("/")[-3])
    if 'external' in path:
        if '.csv' in path:
            external_tsv.append(s3_loader.read_s3_file(path, sep = ",", in_staging=in_staging, profile=profile))
        if '.tsv' in path:
            external_tsv.append(s3_loader.read_s3_file(path, sep = "\t", in_staging=in_staging, profile=profile))

In [9]:
all_load_data_platenames = []
for df in load_data_csvs:
    all_load_data_platenames.extend(df['Metadata_Plate'].unique())
all_barcode_platemap_platenames = []
for df in barcode_platemap_csvs:
    all_barcode_platemap_platenames.extend(df['Plate_Map_Name'].unique())
if set(all_load_data_platenames) != set(all_barcode_platemap_platenames):
    print("Plate names in load_data.csv and barcode_platemap.csv do not match.")
    print('Do NOT proceed until they are matched')
if len(all_load_data_platenames) == 0:
    print("No plate names found in load_data.csv. Please check the load_data.csv files.")

In [10]:
if set(list_batch_names_from_load_data) != set(list_batch_names_from_barcode_platemaps):
    print("Batch names in load_data.csv and barcode_platemap.csv do not match.")
    print('Do NOT proceed until they are matched')
if set(list_batch_names_from_barcode_platemaps) != set(list_batch_names_from_platemaps):
    print("Batch names from barcode_platemap.csv do not match those in platemaps.csv.")
    print('Do NOT proceed until they are matched')

In [11]:
complete_df = project.run_conversion(
    load_data_csvs = load_data_csvs, 
    load_data_csv_batch = list_batch_names_from_load_data, 
    platemap_csvs = barcode_platemap_csvs,  
    platemap_csv_batch = list_batch_names_from_barcode_platemaps, 
    platemap_txt= platemaps, 
    platemap_txt_batch = list_batch_names_from_platemaps,
    platemap_txt_name = list_platemap_names,
    external_tsv = external_tsv,
    external_merge_regex = [["compound"],["compound"]]
)

No external metadata found. Not all projects have external metadata.
Merging in barcode platemap csvs
Merging in platemaps
Skipping external metadata. Not all projects have external metadata.
No concentration columns found — skipping concentration merge.
Harmonizing values of Treatment_Solvent
Harmonizing values of Label


# Additional cleaning steps

In [12]:
# add per-experiment metadata manually annotated above
for col, val in per_dataset_manual.items():
    complete_df[col] = val

# add source information inferred from file path
for source in source_list:
    complete_df.loc[complete_df['File Path'].str.contains(f"/{source}/"),'Source'] = source

In [13]:
with open('../inferable_relationships.json', "r") as f:
    inferable_metadata = json.load(f)

# infer label metadata from known relationships
if per_dataset_manual['CP_Version'] == 'other':
    print("Did not infer label metadata because CP_Version not inferrable")
    print(f"Labels that need to be manually declared are {complete_df['Label'].unique()}")
else:
    found_mismatch = False
    if not all([x in inferable_metadata["Label"].keys() for x in complete_df['Label'].unique()]):
        for x in complete_df['Label'].unique():
            if x not in inferable_metadata["Label"].keys():
                for key, value in inferable_metadata["Label_Alternative_Names"].items():
                    if x in value:
                        print(f"Inferred label {x} to be {key} based on alternative names")
                        complete_df.loc[complete_df['Label'] == x, 'Label'] = key
                        break
                else:
                    # only if there is a label that can't be inferred
                    found_mismatch = True 
    if found_mismatch:
        print(f'Labels need to be corrected to match any of {list(inferable_metadata["Label"].keys())}')
        print(f"Current labels are {complete_df['Label'].unique()}")

# use inferable metadata to fill in missing metadata for each label and cell line
for column in ["Cell_Line_Name", "Label"]:
    for entry in inferable_metadata[column]:
        for inferred_column in inferable_metadata[column][entry]:
            if not inferred_column in complete_df.columns:
                complete_df[inferred_column] = np.nan
                complete_df[inferred_column] = complete_df[inferred_column].astype('str')
            complete_df.loc[complete_df[column]==entry, inferred_column] = inferable_metadata[column][entry][inferred_column]


In [14]:
# add per label excitation and emission values
# can delete if all values are unknown
if set(complete_df['Label'].unique()) == ex_em_fluor_dict.keys():
    for key in ex_em_fluor_dict:
        complete_df.loc[complete_df["Label"] == key, "Microscope_Excitation_Peak"] = ex_em_fluor_dict[key]['ex_peak']
        complete_df.loc[complete_df["Label"] == key, "Microscope_Excitation_Width"] = ex_em_fluor_dict[key]['ex_width']
        complete_df.loc[complete_df["Label"] == key, "Microscope_Emission_Peak"] = ex_em_fluor_dict[key]['em_peak']
        complete_df.loc[complete_df["Label"] == key, "Microscope_Emission_Width"] = ex_em_fluor_dict[key]['em_width']
        complete_df.loc[complete_df["Label"] == key, "Label_Fluorophore"] = ex_em_fluor_dict[key]['fluorophore']
else:
    print("Excitation/Emission dictionary does not match Labels")
    print(f"Available labels are {set(complete_df['Label'].unique())}")
    print(f"Defined keys are {ex_em_fluor_dict.keys()}")

In [15]:
# report on un-harmonized columns
# use reported information to manually update ontology OR dataframe OR input metadata
# this cell does NOT save the harmonization results, just check them
# this allows you to make any necessary corrections without accidentally overwriting
# returns a view of what the dataframe will look like after final harmonization
extra_cols = harmonize_checker.check_columns('../harmonized_ontology.json',complete_df, ret="extra_cols")

print("View of data that will be kept")
complete_df[[x for x in complete_df.columns if x not in extra_cols]]

Column Label_Fluorophore is not harmonized, <StringArray>
['PhenoVue Fluor 555, PhenoVue Fluor 568']
Length: 1, dtype: string not in values
Removing columns not in ontology: ['Label', 'User_Name', 'Plate_Map']
Adding missing ontology columns: ['Image_Position_Z', 'Treatment_Broad_Sample', 'Timepoint_Secondary_Treatment', 'Treatment_Secondary_Treatment', 'Timepoint_Acquisition', 'Treatment_Mechanism', 'Treatment_Control_Class']
View of data that will be kept


,Plate,Well,Site,Batch,File Path,File Name,Treatment_Primary_Treatment,Treatment_Concentration,Treatment_InChIKey,Treatment_PubChem_CID,...,Source,Label_Reagent,Label_Structure,Label_Molecule,Label_Mechanism,Microscope_Excitation_Peak,Microscope_Excitation_Width,Microscope_Emission_Peak,Microscope_Emission_Width,Label_Fluorophore
0,41005680,A01,1,2024_09_20_BSC_Batch1,https://cellpainting-gallery.s3.us-east-1.amaz...,r01c01f01p01-ch1sk1fk1fl1,"6,7,13,14-tetrahydroxy-2,9-dioxatetracyclo[6.6...",33.5,AFSDNFLWKVMVRB-UHFFFAOYSA-N,nan,...,axiom,Hoechst,Nucleus,DNA,Dye,365,30,465,70,33342
1,41005680,A01,2,2024_09_20_BSC_Batch1,https://cellpainting-gallery.s3.us-east-1.amaz...,r01c01f02p01-ch1sk1fk1fl1,"6,7,13,14-tetrahydroxy-2,9-dioxatetracyclo[6.6...",33.5,AFSDNFLWKVMVRB-UHFFFAOYSA-N,nan,...,axiom,Hoechst,Nucleus,DNA,Dye,365,30,465,70,33342
2,41005680,A01,3,2024_09_20_BSC_Batch1,https://cellpainting-gallery.s3.us-east-1.amaz...,r01c01f03p01-ch1sk1fk1fl1,"6,7,13,14-tetrahydroxy-2,9-dioxatetracyclo[6.6...",33.5,AFSDNFLWKVMVRB-UHFFFAOYSA-N,nan,...,axiom,Hoechst,Nucleus,DNA,Dye,365,30,465,70,33342
3,41005680,A01,4,2024_09_20_BSC_Batch1,https://cellpainting-gallery.s3.us-east-1.amaz...,r01c01f04p01-ch1sk1fk1fl1,"6,7,13,14-tetrahydroxy-2,9-dioxatetracyclo[6.6...",33.5,AFSDNFLWKVMVRB-UHFFFAOYSA-N,nan,...,axiom,Hoechst,Nucleus,DNA,Dye,365,30,465,70,33342
4,41005680,A01,5,2024_09_20_BSC_Batch1,https://cellpainting-gallery.s3.us-east-1.amaz...,r01c01f05p01-ch1sk1fk1fl1,"6,7,13,14-tetrahydroxy-2,9-dioxatetracyclo[6.6...",33.5,AFSDNFLWKVMVRB-UHFFFAOYSA-N,nan,...,axiom,Hoechst,Nucleus,DNA,Dye,365,30,465,70,33342
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
276457,41005687,P24,11,2024_09_20_BSC_Batch1,https://cellpainting-gallery.s3.us-east-1.amaz...,r16c24f11p01-ch6sk1fk1fl1,"(2R,3R)-2-(3,4-dihydroxyphenyl)-5,7-dihydroxy-...",0.045643155,LSHVYAFMTMFKBA-TZIWHRDSSA-N,nan,...,axiom,None,None,None,None,740,<NA>,722,75,None
276458,41005687,P24,12,2024_09_20_BSC_Batch1,https://cellpainting-gallery.s3.us-east-1.amaz...,r16c24f12p01-ch6sk1fk1fl1,"(2R,3R)-2-(3,4-dihydroxyphenyl)-5,7-dihydroxy-...",0.045643155,LSHVYAFMTMFKBA-TZIWHRDSSA-N,nan,...,axiom,None,None,None,None,740,<NA>,722,75,None
276459,41005687,P24,13,2024_09_20_BSC_Batch1,https://cellpainting-gallery.s3.us-east-1.amaz...,r16c24f13p01-ch6sk1fk1fl1,"(2R,3R)-2-(3,4-dihydroxyphenyl)-5,7-dihydroxy-...",0.045643155,LSHVYAFMTMFKBA-TZIWHRDSSA-N,nan,...,axiom,None,None,None,None,740,<NA>,722,75,None
276460,41005687,P24,14,2024_09_20_BSC_Batch1,https://cellpainting-gallery.s3.us-east-1.amaz...,r16c24f14p01-ch6sk1fk1fl1,"(2R,3R)-2-(3,4-dihydroxyphenyl)-5,7-dihydroxy-...",0.045643155,LSHVYAFMTMFKBA-TZIWHRDSSA-N,nan,...,axiom,None,None,None,None,740,<NA>,722,75,None


In [16]:
print("View of data that will be removed")
complete_df[extra_cols]

View of data that will be removed


,Label,User_Name,Plate_Map
0,DNA,URL_OrigDNA,41005680
1,DNA,URL_OrigDNA,41005680
2,DNA,URL_OrigDNA,41005680
3,DNA,URL_OrigDNA,41005680
4,DNA,URL_OrigDNA,41005680
...,...,...,...
276457,Brightfield,URL_OrigBrightfield,41005687
276458,Brightfield,URL_OrigBrightfield,41005687
276459,Brightfield,URL_OrigBrightfield,41005687
276460,Brightfield,URL_OrigBrightfield,41005687


# FINAL DATA CLEANUP

In [17]:
# After correcting any warnings above, run to save harmonization
complete_df = harmonize_checker.check_columns('../harmonized_ontology.json',complete_df)

Column Label_Fluorophore is not harmonized, <StringArray>
['PhenoVue Fluor 555, PhenoVue Fluor 568']
Length: 1, dtype: string not in values
Removing columns not in ontology: ['Label', 'User_Name', 'Plate_Map']
Adding missing ontology columns: ['Image_Position_Z', 'Treatment_Broad_Sample', 'Timepoint_Secondary_Treatment', 'Treatment_Secondary_Treatment', 'Timepoint_Acquisition', 'Treatment_Mechanism', 'Treatment_Control_Class']


In [18]:
saved_path = os.path.join(output_directory,f"{project_name}_harmonized_metadata_v0_1.parquet")
complete_df.to_parquet(saved_path)